In [1]:
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv("data/student_records.csv")
df.head()

,index,number_of_siblings,direct_admission,CCA,learning_style,student_id,gender,tuition,final_test,n_male,n_female,age,hours_per_week,attendance_rate,sleep_time,wake_time,mode_of_transport,bag_color
0,0,0,Yes,Sports,Visual,ACN2BE,Female,No,69.0,14.0,2.0,16.0,10.0,91.0,22:00,6:00,private transport,yellow
1,1,2,No,Sports,Auditory,FGXIIZ,Female,No,47.0,4.0,19.0,16.0,7.0,94.0,22:30,6:30,private transport,green
2,2,0,Yes,NaN,Visual,B9AI9F,Male,No,85.0,14.0,2.0,15.0,8.0,92.0,22:30,6:30,private transport,white
3,3,1,No,Clubs,Auditory,FEVM1T,Female,Yes,64.0,2.0,20.0,15.0,18.0,NaN,21:00,5:00,public transport,yellow
4,4,0,No,Sports,Auditory,AXZN2E,Male,No,66.0,24.0,3.0,16.0,7.0,95.0,21:30,5:30,public transport,yellow


## Data Cleaning

### Removing Duplicate Records

During the EDA, 900 students were found to appear twice in the dataset, with identical records apart from `bag_color` and some missing values. Before removing the duplicates, any missing `final_test` or `attendance_rate` in one row is filled using the duplicate row with the same `student_id`, so that no real values are lost. The duplicates are then removed so that each student appears only once, which also prevents the same student from ending up in both the training and test sets.

In [3]:
print('Shape before removing duplicates:', df.shape)
print('Duplicate student_id rows:', df['student_id'].duplicated().sum())
print('\nMissing before recovery:')
print(df[['final_test', 'attendance_rate']].isna().sum())

# Recover missing values from each student's duplicate row
for col in ['final_test', 'attendance_rate']:
    df[col] = df.groupby('student_id')[col].transform(
        lambda s: s.fillna(s.dropna().iloc[0]) if s.notna().any() else s
    )

# Remove duplicates, keeping the first record for each student
df = df.drop_duplicates(subset='student_id', keep='first').reset_index(drop=True)

# Verify the changes
print('\nShape after removing duplicates:', df.shape)
print('Duplicate student_id rows remaining:', df['student_id'].duplicated().sum())
print('\nMissing after recovery and deduplication:')
print(df[['final_test', 'attendance_rate']].isna().sum())

Shape before removing duplicates: (15900, 18)
Duplicate student_id rows: 900

Missing before recovery:
final_test         495
attendance_rate    778
dtype: int64

Shape after removing duplicates: (15000, 18)
Duplicate student_id rows remaining: 0

Missing after recovery and deduplication:
final_test         441
attendance_rate    688
dtype: int64


### Handling Missing Values

After recovering values from duplicate rows, 441 rows still have a missing `final_test` and 688 have a missing `attendance_rate`.

For `final_test`, the remaining rows were dropped rather than imputed. As it is the target the model is trying to predict, filling it in would mean training and evaluating the model on made-up scores rather than real ones. Filling every missing value with the same number, such as the median, would also pull the model's predictions towards the middle. As these rows make up only about 3% of the data, dropping them costs very little.

For `attendance_rate`, the missing values are left as they are for now. They will be filled with the median inside the preprocessing pipeline during model development, so that the median is calculated from the training set only. Calculating it from the whole dataset before splitting would let information from the validation and test sets leak into training. The median was chosen over the mean because it is not affected by extreme values, making it a better reflection of a typical student. This is especially important for `attendance_rate`, where most students have high attendance and a small number of low values would pull the mean down.

In [4]:
print('Shape before dropping missing final_test:', df.shape)

# Drop rows where the target is missing
df = df.dropna(subset=['final_test']).reset_index(drop=True)

# Verify the changes (attendance_rate is imputed later in the preprocessing pipeline)
print('Shape after dropping missing final_test:', df.shape)
print(df[['final_test', 'attendance_rate']].isna().sum())

Shape before dropping missing final_test: (15000, 18)
Shape after dropping missing final_test: (14559, 18)
final_test           0
attendance_rate    669
dtype: int64


### Correcting Invalid Ages

The EDA found values in `age` that are not valid for students taking O-level Mathematics in Singapore, where the expected age is 15 or 16. The column contains single-digit ages (5 and 6) and negative ages (-5 and -4).

Ages 5 and 6 were assumed to be data entry errors with the leading "1" left out, and were corrected to 15 and 16. As the correct values for the negative ages cannot be determined, they were imputed with the mode of the valid ages. Although `age` is dropped later, it is cleaned first so that the dataset is accurate at every stage.

In [5]:
# Count invalid ages before fixing
invalid_age = ~df['age'].isin([15, 16])
print('Number of invalid ages:', invalid_age.sum())
print(df.loc[invalid_age, 'age'].value_counts())

# Fix data entry errors
df['age'] = df['age'].replace({5: 15, 6: 16})

# Impute negative ages with the mode of valid ages
mode_age = df.loc[df['age'] > 0, 'age'].mode()[0]
df.loc[df['age'] < 0, 'age'] = mode_age
print('\nMode age used for imputation:', mode_age)

# Verify the changes
print('Invalid ages remaining:', (~df['age'].isin([15, 16])).sum())

Number of invalid ages: 420
age
 6.0    216
 5.0    199
-5.0      4
-4.0      1
Name: count, dtype: int64

Mode age used for imputation: 16.0
Invalid ages remaining: 0


### Standardising Categorical Labels

`tuition` contains both "Yes" and "Y", as well as "No" and "N", which have the same meaning. This was assumed to be a labelling inconsistency during data entry, so the values were standardised to "Yes" and "No".

In [6]:
# Standardise tuition labels
df['tuition'] = df['tuition'].replace({'Y': 'Yes', 'N': 'No'})

# Verify the changes
print(df['tuition'].value_counts())

tuition
Yes    8252
No     6307
Name: count, dtype: int64


`CCA` has inconsistent capitalisation, where "Sports" and "SPORTS", "Arts" and "ARTS", and "Clubs" and "CLUBS" refer to the same CCA types. These were standardised to title case.

The EDA also showed that students with a missing `CCA` have almost the same profile as those labelled "NONE", with a similar average `final_test`, attendance rate and study hours. Both groups were therefore combined into a single "No CCA" category.

In [7]:
# Combine missing values with 'None', standardise to title case, then rename
df['CCA'] = df['CCA'].fillna('None').str.title()
df['CCA'] = df['CCA'].replace('None', 'No CCA')

# Verify the changes
print(df['CCA'].value_counts())

CCA
Clubs     3707
Sports    3641
No CCA    3617
Arts      3594
Name: count, dtype: int64


## Feature Engineering

### Feature Engineering - 'class_size'

`n_male` and `n_female` were combined into `class_size`, as the total number of students in a class has a much stronger relationship with `final_test` than either count on its own.

In [8]:
# Combine the number of male and female students into class size
df['class_size'] = df['n_male'] + df['n_female']

# Verify the new feature
print('Missing values in class_size:', df['class_size'].isna().sum())
df['class_size'].describe()

Missing values in class_size: 0


count    14559.000000
mean        22.778350
std          4.234682
min         14.000000
25%         20.000000
50%         22.000000
75%         27.000000
max         31.000000
Name: class_size, dtype: float64

### Feature Engineering - 'sleep_hours'

`sleep_time` and `wake_time` were combined into `sleep_hours`. Where bedtime is before midnight, direct subtraction gives a negative result, so 24 hours is added to correct it.

In [9]:
# Convert sleep_time and wake_time from strings to datetime
sleep = pd.to_datetime(df['sleep_time'], format='%H:%M')
wake = pd.to_datetime(df['wake_time'], format='%H:%M')

# Calculate sleep duration, adding a day when bedtime is before midnight
sleep_duration = wake - sleep
sleep_duration = sleep_duration.where(sleep_duration >= pd.Timedelta(0), sleep_duration + pd.Timedelta(days=1))

# Convert the duration to hours
df['sleep_hours'] = sleep_duration.dt.total_seconds() / 3600

# Verify with one random student for each sleep duration
display(df.groupby('sleep_hours')[['sleep_time', 'wake_time', 'sleep_hours']].sample(1, random_state=42))
print('Missing values in sleep_hours:', df['sleep_hours'].isna().sum())

,sleep_time,wake_time,sleep_hours
8207,1:30,5:30,4.0
8854,2:00,7:00,5.0
6086,23:00,5:00,6.0
5727,23:00,6:00,7.0
2811,21:00,5:00,8.0


Missing values in sleep_hours: 0


## Dropping Irrelevant Features

`index` and `student_id` were dropped as they are identifiers rather than features. The cardinality check in the EDA showed that every row has a unique value in both columns, and neither has any relationship with `final_test`. Keeping them would add noise and, if encoded, create thousands of meaningless columns.

`gender`, `n_male`, `n_female`, `age`, `sleep_time`, `wake_time`, `mode_of_transport` and `bag_color` were also dropped, either because they show no relationship with `final_test` or because they have been replaced by more useful engineered features:

- **No relationship with `final_test`:** the box plots for `gender`, `mode_of_transport` and `bag_color` show almost identical medians and spreads across all groups, and statistical tests in the EDA confirmed there is no significant difference in `final_test` between their groups (p > 0.05). `age` also has no correlation with `final_test` in the Spearman correlation matrix.
- **Replaced by `class_size`:** `n_male` and `n_female` are only weakly correlated with `final_test` on their own, but combining them into `class_size` gives a much stronger relationship. Keeping all three would also cause multicollinearity, since `class_size` is calculated directly from them.
- **Replaced by `sleep_hours`:** `sleep_time` and `wake_time` were combined into `sleep_hours`, which captures the useful information from both in a single feature.

In [10]:
# Drop identifiers, features with no relationship to final_test, and features replaced by engineered ones
cols_to_drop = ['index', 'student_id', 'gender', 'n_male', 'n_female', 'age',
                'sleep_time', 'wake_time', 'mode_of_transport', 'bag_color']
df = df.drop(columns=cols_to_drop)

# Verify the remaining columns
print('Remaining columns:', df.columns.tolist())

Remaining columns: ['number_of_siblings', 'direct_admission', 'CCA', 'learning_style', 'tuition', 'final_test', 'hours_per_week', 'attendance_rate', 'class_size', 'sleep_hours']


## Handling Multicollinearity

As identified in the EDA, `attendance_rate` and `sleep_hours` are related, as students who sleep more tend to have better attendance. The VIF is recalculated here to confirm that it is still below the commonly used threshold of 5 after cleaning.

In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

vif_features = ['number_of_siblings', 'class_size', 'hours_per_week', 'attendance_rate', 'sleep_hours']

# VIF cannot handle missing values, so rows with a missing attendance_rate are skipped
X = add_constant(df[vif_features].dropna())

vif = pd.DataFrame({
    'feature': vif_features,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
}).round(2)

vif

,feature,VIF
0,number_of_siblings,1.04
1,class_size,1.05
2,hours_per_week,1.05
3,attendance_rate,4.72
4,sleep_hours,4.66


All features have a VIF below 5. `attendance_rate` and `sleep_hours` remain at around 4.7, so both features are kept, although their coefficients in a linear regression model should be interpreted with caution.

## Converting to Binary Features

`direct_admission`, `tuition` and `learning_style` each have only two categories, so they were converted to 1 and 0. This is equivalent to one-hot encoding with one column dropped, and keeps each feature as a single column.

In [12]:
df['direct_admission'] = df['direct_admission'].map({'Yes': 1, 'No': 0})
df['tuition'] = df['tuition'].map({'Yes': 1, 'No': 0})
df['learning_style'] = df['learning_style'].map({'Visual': 1, 'Auditory': 0})

# Verify that no values were lost in the conversion
print(df[['direct_admission', 'tuition', 'learning_style']].isna().sum())

direct_admission    0
tuition             0
learning_style      0
dtype: int64


## Final Check

In [13]:
# Final check before saving
print('Final shape:', df.shape)
print('\nMissing values:')
print(df.isna().sum())
df.head()

Final shape: (14559, 10)

Missing values:
number_of_siblings      0
direct_admission        0
CCA                     0
learning_style          0
tuition                 0
final_test              0
hours_per_week          0
attendance_rate       669
class_size              0
sleep_hours             0
dtype: int64


,number_of_siblings,direct_admission,CCA,learning_style,tuition,final_test,hours_per_week,attendance_rate,class_size,sleep_hours
0,0,1,Sports,1,0,69.0,10.0,91.0,16.0,8.0
1,2,0,Sports,0,0,47.0,7.0,94.0,23.0,8.0
2,0,1,No CCA,1,0,85.0,8.0,92.0,16.0,8.0
3,1,0,Clubs,0,1,64.0,18.0,NaN,22.0,8.0
4,0,0,Sports,0,0,66.0,7.0,95.0,27.0,8.0


The cleaned dataset has 14,559 students and 10 columns. The only remaining missing values are in `attendance_rate`, which will be imputed with the median inside the preprocessing pipeline.

In [14]:
df.to_csv("data/cleaned_student_records.csv", index=False)